In [1]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_Sil2Gld_DimFiles V2"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Silver/Dim_Files" # ← Change source path
TARGET_PATH = "abfss://Gold/Dim_Files" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_gold_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_gold_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_gold_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 3, Finished, Available, Finished)

🔧 Initializing ntk_Sil2Gld_DimFiles V2...
🚀 Starting ntk_Sil2Gld_DimFiles V2


In [2]:

from pyspark.sql import SparkSession
from datetime import datetime
import os

# Base source path (up to Files level)
base_source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files"
# Base target path  
base_target_path  = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files"

# Get today's date and format it
today = datetime.now()  
from datetime import datetime, timedelta
today = today ##- timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d")  

# Build dynamic source folder structure
sourcefolder_structure = f"Silver_layer/Reporting/{year}/{month}/{day}"   
# Complete source path
complete_source_path = f"{base_source_path}/{sourcefolder_structure}"
# Filename
source_filename = "Dim_Files.parquet"

# Build dynamic target folder structure
targetfolder_structure = f"PreGold_Reporting"   
# Complete target path
complete_target_path = f"{base_target_path}/{targetfolder_structure}"
# Filename
target_filename = "Dim_Files.parquet"
print(f"Variables created and session started.")


StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 4, Finished, Available, Finished)

Variables created and session started.


In [3]:

# Initialize Spark session
spark = SparkSession.builder.appName("SilverToGold_FileReader").getOrCreate()

# Full Source  file path
full_source_path = f"{complete_source_path}/{source_filename}"

# Full Target  file path
full_target_path = f"{complete_target_path}/{target_filename}"

print(f"Source: {full_source_path}")
print(f"Target: {full_target_path}")


StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 5, Finished, Available, Finished)

Source: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/15/Dim_Files.parquet
Target: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files/PreGold_Reporting/Dim_Files.parquet


In [4]:
# mssparkutils.notebook.run("nbk_dimuser_validations", 60)

StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 6, Finished, Available, Finished)

In [5]:
# # Reading source and target data to dataframes.

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, col, when

# columns_needed = ["User_GUIDPK","UserPrincipalName","Email"]
# Read Parquet files directly into memory
print("Loading Parquet files into memory...")
source_df = spark.read.parquet(full_source_path)

# source_df = source_df.select(*columns_needed)
source_df.printSchema()

target_df = spark.read.parquet(full_target_path)

# try:
#     target_df = spark.read.parquet(full_target_path)
# except:
    # # Create empty DataFrame with same schema if target doesn't exist
    # target_df = spark.createDataFrame([], source_df.schema)

# columns_needed = ["UserKey","UserPrincipalName","Email"]
# target_df = target_df.select(*columns_needed)
target_df.printSchema()

print(f"Reading source file completed {full_source_path}")
print(f"Reading target file completed {full_target_path}")

StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 7, Finished, Available, Finished)

Loading Parquet files into memory...
root
 |-- LibraryName: string (nullable = true)
 |-- LibraryUrl: string (nullable = true)
 |-- FileName: string (nullable = true)
 |-- FileExtension: string (nullable = true)
 |-- FileType: string (nullable = true)
 |-- UniqueId: string (nullable = true)
 |-- ItemId: integer (nullable = true)
 |-- GUID: string (nullable = true)
 |-- SiteId: string (nullable = true)
 |-- SiteName: string (nullable = true)
 |-- SiteUrl: string (nullable = true)
 |-- FilePath: string (nullable = true)
 |-- FolderPath: string (nullable = true)
 |-- FileUrl: string (nullable = true)
 |-- FileSizeBytes: integer (nullable = true)
 |-- FileSizeKB: double (nullable = true)
 |-- FileSizeMB: double (nullable = true)
 |-- FileSizeFormatted: string (nullable = true)
 |-- Created: string (nullable = true)
 |-- Modified: string (nullable = true)
 |-- CreatedBy: string (nullable = true)
 |-- CreatedByEmail: string (nullable = true)
 |-- LastModifiedBy: string (nullable = true)
 |--

In [6]:
# Cache for performance
target_df.cache()
print(target_df.count())

StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 8, Finished, Available, Finished)

2162


In [7]:
# # Fucntion for Calibrating source column definations to target column schema
from pyspark.sql.functions import col
from pyspark.sql.types import *

def align_to_target_schema_only(source_df, target_df, source_columns, target_columns):
    """  Align to match ONLY target schema (lose extra source columns)  """
    
    # Get target schema for type casting
    target_schema = {field.name: field.dataType for field in target_df.schema.fields}
    
    # Start with source DataFrame
    aligned_df = source_df.select(source_columns)
    target_df = target_df.select(target_columns)
    mapped_target_cols = set()

    # # STEP 1: Rename mapped columns (A→X, B→Y, C→Z)
    print(f"\n🔄 STEP 1: Renaming {len(source_columns)} mapped columns...")
    for src_col, tgt_col in zip(source_columns, target_columns):
        if src_col in aligned_df.columns:
            if src_col != tgt_col:  # Only rename if different
                aligned_df = aligned_df.withColumnRenamed(src_col, tgt_col)
                # print(f"  📝 {src_col} → {tgt_col}")
            # else:
                # print(f"  ✅ {src_col} (already correct name)")
            mapped_target_cols.add(tgt_col)
       # else:
            # print(f"  ⚠️ Source column '{src_col}' not found in source DF")
    
   # print(f"   After renaming: {aligned_df.columns}")
    
    # # STEP 2: Cast mapped columns to target types
    print(f"\n🔧 STEP 2: Casting {len(mapped_target_cols)} mapped columns...")
    for tgt_col in mapped_target_cols:
        if tgt_col in aligned_df.columns:
            target_type = target_schema.get(tgt_col, StringType())
            aligned_df = aligned_df.withColumn(tgt_col, col(tgt_col).cast(target_type))
           # print(f"  🔧 {tgt_col} cast to {target_type}")
   
    return aligned_df

StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 9, Finished, Available, Finished)

In [8]:
# # Updating / triming source and target dataframes to required columns & renaming source columns to target column mapping 

source_cols = ["FileKey", "LibraryKey", "SiteKey", "UniqueId", "ItemId", "SiteId", "FileName", "FileUrl", "FileExtension", "FileType", "ContentType", "Owner", "DataRedundancy", 
"Version", "IsCurrentVersion", "SensitivityLabel", "Classification", "ComplianceTag", "ApprovalStatus", "IsSharedExternally",]

target_cols = ["FileKey", "LibraryKey", "SiteKey", "ObjectID", "ItemID", "SiteID", "FileName", "FilePath", "FileExtension", "FileType", "ContentType", "Owner", 
"DataRedundancy", "Version", "IsCurrentVersion","SensitivityLabel", "Classification", "ComplianceTag", "ApprovalStatus", "IsSharedExternally"]


# upd_source_df = source_df.filter(col("PrincipalType") == "User").select(source_cols)
upd_source_df = source_df.select(source_cols)
upd_target_df = target_df.select(target_cols)
# print(f"Source Schema: {upd_source_df.printSchema()}")
# print(f"Target Schema: {upd_target_df.printSchema()}")
upd_source_df = align_to_target_schema_only(upd_source_df, upd_target_df, source_cols, target_cols)

# print(f"Source schema  beefore:")
# print(f"{source_df.printSchema()}")
print(f"Source post remapping:")
print(f"{upd_source_df.printSchema()}")


StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 10, Finished, Available, Finished)


🔄 STEP 1: Renaming 20 mapped columns...

🔧 STEP 2: Casting 20 mapped columns...
Source post remapping:
root
 |-- FileKey: string (nullable = true)
 |-- LibraryKey: string (nullable = true)
 |-- SiteKey: string (nullable = true)
 |-- ObjectID: string (nullable = true)
 |-- ItemID: integer (nullable = true)
 |-- SiteID: string (nullable = true)
 |-- FileName: string (nullable = true)
 |-- FilePath: string (nullable = true)
 |-- FileExtension: string (nullable = true)
 |-- FileType: string (nullable = true)
 |-- ContentType: string (nullable = true)
 |-- Owner: string (nullable = true)
 |-- DataRedundancy: string (nullable = true)
 |-- Version: double (nullable = true)
 |-- IsCurrentVersion: boolean (nullable = true)
 |-- SensitivityLabel: string (nullable = true)
 |-- Classification: string (nullable = true)
 |-- ComplianceTag: string (nullable = true)
 |-- ApprovalStatus: string (nullable = true)
 |-- IsSharedExternally: boolean (nullable = true)

None


In [9]:
from pyspark.sql.functions import col, current_timestamp

#     Detect INSERT records (rows in source not in target)
def detect_records2_inserts(source_df, target_df, sourcekey, targetkey):
    
    # Cache for performance
    source_df.cache()
    target_df.cache()
    
    # Build join condition (source.key = target.key)
    join_condition = col(f"s.{sourcekey}") == col(f"t.{targetkey}")
    
    # Find records in source not in target
    inserts_df = (
        source_df.alias("s")
        .join(target_df.alias("t"), join_condition, "left_anti")
        .withColumn("ModifiedDate", current_timestamp())
    )
    
    insert_count = inserts_df.count()
    print(f"🆕 INSERT: {insert_count} records found")
    
    return inserts_df.select(sourcekey)

def detect_records2updates(source_df, target_df, sourcekey, targetkey):
    
    # Cache for performance
    source_df.cache()
    target_df.cache()
    
    # Build join condition
    join_condition = col(f"s.{sourcekey}") == col(f"t.{targetkey}")
    
    # Get non-key columns for comparison
    key_column = sourcekey
    non_key_cols = [c for c in source_df.columns if c != key_column]
    
    # Build dynamic "any column differs" filter
    diff_condition = None
    for c in non_key_cols:
        cond = col(f"s.{c}") != col(f"t.{c}")
        diff_condition = cond if diff_condition is None else (diff_condition | cond)
    
    # Find records that exist in both but have differences
    updates_df = (
        source_df.alias("s")
        .join(target_df.alias("t"), join_condition, "inner")
        .filter(diff_condition)  # Only records with differences
        .select(
            col("s.*"),  # Take updated values from source
            current_timestamp().alias("ModifiedDate")
        )
    )
    
    update_count = updates_df.count()
    print(f"✏️ UPDATE: {update_count} records found")
    
    return updates_df.select(sourcekey)

    # Detect DELETE records (rows in target not in source)    
def detect_records2_deletes(source_df, target_df, sourcekey, targetkey):
    
    # Cache for performance
    source_df.cache()
    target_df.cache()
    
    # Build join condition (target.key = source.key) 
    join_condition = col(f"t.{targetkey}") == col(f"s.{sourcekey}")
    
    # Find records in target not in source
    deletes_df = (
        target_df.alias("t")
        .join(source_df.alias("s"), join_condition, "left_anti")
        .withColumn("LastModified", current_timestamp())
    )
    
    delete_count = deletes_df.count()
    print(f"🗑️ DELETE: {delete_count} records found")
    
    return deletes_df.select(targetkey)


StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 11, Finished, Available, Finished)

In [10]:
# # Code to check new records and add to dataframe *****

from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp, col
from pyspark.sql.types import *

# # Method 1a: Create single new record using Row
# upd_source_df.show(50)
new_record = spark.createDataFrame([
    Row(
            FileKey = "123",
            LibraryKey = "lib456",
            SiteKey = "site789",
            ObjectID = "obj001",
            ItemID = "itm002",
            SiteID = "s001",
            FileName = "Report_Q3.xlsx",
            FilePath = "/sites/finance/docs/Report_Q3.xlsx",
            FileExtension = ".xlsx",
            FileType = "Document",
            ContentType = "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
            Owner = "john.doe@company.com",
            DataRedundancy = "GeoRedundant",
            Version = "3.1",
            IsCurrentVersion = True,
            SensitivityLabel = "Confidential",
            Classification = "Finance",
            ComplianceTag = "Retention_7Y",
            ApprovalStatus = "Approved",
            IsSharedExternally = False
    )
])

upd_source_df = upd_source_df.union(new_record)
new_result_df = detect_records2_inserts(upd_source_df, upd_target_df,"FileKey","FileKey")
# print(f"target before:{target_df.count()}")

# Step 1: Join source with new_df to get complete records for new keys
# source_cols =  ["FileKey", "LibraryKey", "SiteKey", "UniqueId", "ItemId", "SiteId", "LibraryName", "FileName", "FileUrl", "FileExtension", "FileType", "ContentType", "Owner", "DataRedundancy", 
# "Version", "IsCurrentVersion", "SensitivityLabel", "Classification", "ComplianceTag", "ApprovalStatus", "IsSharedExternally", "CreatedByEmail", "LastModifiedBy", "LastModifiedDate"]
# target_cols = ["FileKey", "LibraryKey", "SiteKey", "ObjectID", "ItemID", "SiteID", "DocumentLibraryID", "FileName", "FilePath", "FileExtension", "FileType", "ContentType", "Owner", 
# "DataRedundancy", "Version", "IsCurrentVersion","SensitivityLabel", "Classification", "ComplianceTag", "ApprovalStatus", "IsSharedExternally", "CreatedBy", "CreatedDate", "ModifiedBy", "ModifiedDate"]

# Create column mapping (source -> target)
column_mapping = dict(zip(target_cols, target_cols))
new_records_df = (
    new_result_df.alias("n")
    .join(upd_source_df.alias("s"), col("n.FileKey") == col("s.FileKey"), "inner")
    .select(*[col(f"s.{source_cols}").alias(target_cols) for source_cols, target_cols in column_mapping.items()])
)

new_records_df.show()
# print(f"target before:{new_records.count()}")


StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 12, Finished, Available, Finished)

🆕 INSERT: 4 records found
+--------------------+--------------------+--------------------+--------------------+------+--------------------+--------------------+--------------------+-------------+-----------+--------------------+--------------------+--------------+-------+----------------+----------------+--------------+-------------+--------------+------------------+
|             FileKey|          LibraryKey|             SiteKey|            ObjectID|ItemID|              SiteID|            FileName|            FilePath|FileExtension|   FileType|         ContentType|               Owner|DataRedundancy|Version|IsCurrentVersion|SensitivityLabel|Classification|ComplianceTag|ApprovalStatus|IsSharedExternally|
+--------------------+--------------------+--------------------+--------------------+------+--------------------+--------------------+--------------------+-------------+-----------+--------------------+--------------------+--------------+-------+----------------+----------------+------

In [11]:
# # Code to check removed record from source and remove from target table dataframe *****

from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp, col
from pyspark.sql.types import *

# Delete Record in users 

# # Remove specific user by UserPrincipalName
upd_source_df = upd_source_df.filter(col("FileName") != "MSRA_MSTeamsEss_Seg_01-12-2023.xlsx")
# upd_source_df.filter(col("UserPrincipalName") == "sureshgupta@peopletech.com").show()

# # Perform removal of deleted records - in-memory merge
delete_result_df = detect_records2_deletes(upd_source_df, upd_target_df,"FileKey","FileKey")
delete_result_df.show(2)

# upd_target_df.filter(col("UserPrincipalName") == "sureshgupta@peopletech.com").show(25)
# upd_target_df= upd_target_df.join(delete_result_df.select("UserKey"), "UserKey", "left_anti")
# upd_target_df.filter(col("UserPrincipalName") == "sureshgupta@peopletech.com").show(25)

print(f"Source deleted record has been locted")

StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 13, Finished, Available, Finished)

🗑️ DELETE: 6 records found
+--------------------+
|             FileKey|
+--------------------+
|66e8c51ba4f90493b...|
|92aef7b7a7bee6154...|
+--------------------+
only showing top 2 rows

Source deleted record has been locted


In [12]:
# # Code to check updated records and add to dataframe *****

from pyspark.sql import Row
from pyspark.sql.functions import * 
# import lit, current_timestamp, col,when, upper, trim
from pyspark.sql.types import *

# # Update Department for all users in "Software Engineering" 
upd_source_df.filter(col("ItemID") == "572").show(2)

upd_source_df = upd_source_df.withColumn(
    "Owner",
    when(col("ItemID") == "572", "Venkatesh Rao")
    .otherwise(col("Owner"))
)

upd_source_df.show(2)

source_cols = ["FileKey", "LibraryKey", "SiteKey", "UniqueId", "ItemId", "SiteId", "FileName", "FileUrl", "FileExtension", "FileType", "ContentType", "Owner", "DataRedundancy", 
                "Version", "IsCurrentVersion", "SensitivityLabel", "Classification", "ComplianceTag", "ApprovalStatus", "IsSharedExternally",]
target_cols = ["FileKey", "LibraryKey", "SiteKey", "ObjectID", "ItemID", "SiteID", "FileName", "FilePath", "FileExtension", "FileType", "ContentType", "Owner", 
                "DataRedundancy", "Version", "IsCurrentVersion","SensitivityLabel", "Classification", "ComplianceTag", "ApprovalStatus", "IsSharedExternally"]

# source_cols = ["FileKey", "LibraryKey", "SiteKey", "UniqueId", "ItemId", "SiteId"]
# target_cols = ["FileKey", "LibraryKey", "SiteKey", "ObjectID", "ItemID", "SiteID"]

upd_result_df = detect_records2updates(upd_source_df, upd_target_df,"FileKey","FileKey")
upd_result_df.show()



StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 14, Finished, Available, Finished)

+--------------------+--------------------+--------------------+--------------------+------+--------------------+--------------------+--------------------+-------------+-----------+-----------+-----------+--------------+-------+----------------+----------------+--------------+-------------+--------------+------------------+
|             FileKey|          LibraryKey|             SiteKey|            ObjectID|ItemID|              SiteID|            FileName|            FilePath|FileExtension|   FileType|ContentType|      Owner|DataRedundancy|Version|IsCurrentVersion|SensitivityLabel|Classification|ComplianceTag|ApprovalStatus|IsSharedExternally|
+--------------------+--------------------+--------------------+--------------------+------+--------------------+--------------------+--------------------+-------------+-----------+-----------+-----------+--------------+-------+----------------+----------------+--------------+-------------+--------------+------------------+
|68822d5be5994fcda...|

In [13]:
target_df.printSchema()

StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 15, Finished, Available, Finished)

root
 |-- FileKey: string (nullable = true)
 |-- LibraryKey: string (nullable = true)
 |-- SiteKey: string (nullable = true)
 |-- ObjectID: string (nullable = true)
 |-- ItemID: integer (nullable = true)
 |-- SiteID: string (nullable = true)
 |-- FileName: string (nullable = true)
 |-- FilePath: string (nullable = true)
 |-- FileExtension: string (nullable = true)
 |-- FileType: string (nullable = true)
 |-- ContentType: string (nullable = true)
 |-- Owner: string (nullable = true)
 |-- DCLocation: string (nullable = true)
 |-- GeoLocation: string (nullable = true)
 |-- DataRedundancy: string (nullable = true)
 |-- Version: double (nullable = true)
 |-- IsCurrentVersion: boolean (nullable = true)
 |-- SensitivityLabel: string (nullable = true)
 |-- Classification: string (nullable = true)
 |-- ComplianceTag: string (nullable = true)
 |-- RetentionPolicy: string (nullable = true)
 |-- IsActive: string (nullable = true)
 |-- ApprovalStatus: string (nullable = true)
 |-- IsSharedExternall

In [14]:
# # # Code to update target dataframe with updated / inserrted / deleted records *************8

# new_records_df.show()
# upd_result_df.show()
final_result_Key = (
    new_records_df.select(col("FileKey").alias("FileKey"))
    .union(upd_result_df.select(col("FileKey").alias("FileKey")))
)
###final_result_Key.show()

### Get updated records with specified columns only
source_cols = ["FileKey", "LibraryKey", "SiteKey", "UniqueId", "ItemId", "SiteId", "FileName", "FileUrl", "FileExtension", "FileType", "ContentType", "Owner", "DataRedundancy", 
"Version", "IsCurrentVersion", "SensitivityLabel", "Classification", "ComplianceTag", "ApprovalStatus", "IsSharedExternally",]
target_cols = ["FileKey", "LibraryKey", "SiteKey", "ObjectID", "ItemID", "SiteID", "FileName", "FilePath", "FileExtension", "FileType", "ContentType", "Owner", 
"DataRedundancy", "Version", "IsCurrentVersion","SensitivityLabel", "Classification", "ComplianceTag", "ApprovalStatus", "IsSharedExternally"]

updated_add_records = (
    final_result_Key.alias("n")
    .join(source_df.alias("s"), col("n.FileKey") == col("s.FileKey") , "inner")
    .select(*[f"s.{col}" for col in source_cols])
)
updated_add_records.show(3)
updated_add_records = target_df.withColumn("IsDeleted", lit(0))
updated_add_records.show(3)


StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 16, Finished, Available, Finished)

+--------------------+--------------------+--------------------+--------------------+------+--------------------+--------------------+--------------------+-------------+-----------+-----------+------------+--------------+-------+----------------+----------------+--------------+-------------+--------------+------------------+
|             FileKey|          LibraryKey|             SiteKey|            UniqueId|ItemId|              SiteId|            FileName|             FileUrl|FileExtension|   FileType|ContentType|       Owner|DataRedundancy|Version|IsCurrentVersion|SensitivityLabel|Classification|ComplianceTag|ApprovalStatus|IsSharedExternally|
+--------------------+--------------------+--------------------+--------------------+------+--------------------+--------------------+--------------------+-------------+-----------+-----------+------------+--------------+-------+----------------+----------------+--------------+-------------+--------------+------------------+
|e2bc91d498dcefbeb.

In [15]:
# # code to creeate delete recor dataframe and update isdeleted to 1 and other system values.
source_cols = ["FileKey", "LibraryKey", "SiteKey", "UniqueId", "ItemId", "SiteId", "FileName", "FileUrl", "FileExtension", "FileType", "ContentType", "Owner", "DataRedundancy", 
            "Version", "IsCurrentVersion", "SensitivityLabel", "Classification", "ComplianceTag", "ApprovalStatus", "IsSharedExternally",]
target_cols = ["FileKey", "LibraryKey", "SiteKey", "ObjectID", "ItemID", "SiteID", "FileName", "FilePath", "FileExtension", "FileType", "ContentType", "Owner", 
            "DataRedundancy", "Version", "IsCurrentVersion","SensitivityLabel", "Classification", "ComplianceTag", "ApprovalStatus", "IsSharedExternally"]

updated_delrecords = (
    delete_result_df.alias("n")
    .join(upd_target_df.alias("s"), col("n.FileKey") == col("s.FileKey") , "inner")
    .select(*[f"s.{col}" for col in target_cols])
)

updated_delrecords = updated_delrecords .withColumn("IsDeleted", lit(1))
updated_delrecords.show(0)
updated_add_records.show(0)

# Join on FileKey (or any other appropriate common key)
final_target_rec = updated_add_records.join(updated_delrecords, on="FileKey", how="inner")  # or "left", "outer", etc.

# final_target_rec.show()
# final_target_rec.printSchema
# df = df.drop("IsDeleted")
from pyspark.sql import functions as F

# 1️⃣ List of columns to keep from updated_add_records (source)
columns_to_keep = [
    "FileKey", "LibraryKey", "SiteKey", "ObjectID", "ItemID", "SiteID",
    "FileName", "FilePath", "FileExtension", "FileType", "ContentType",
    "Owner", "DCLocation", "GeoLocation", "DataRedundancy", "Version",
    "IsCurrentVersion", "SensitivityLabel", "Classification", "ComplianceTag",
    "RetentionPolicy", "IsActive", "ApprovalStatus", "IsSharedExternally",
    "CreatedBy", "CreatedDate", "ModifiedBy", "ModifiedDate", "SnapshotDate",
    "IsDeleted"
]

# 2️⃣ Rename conflicting columns in updated_delrecords
dup_cols = [c for c in updated_delrecords.columns if c in columns_to_keep and c != "FileKey"]
updated_delrecords_renamed = updated_delrecords.select(
    [F.col("FileKey")] +
    [F.col(c).alias(c + "_del") if c in dup_cols else F.col(c) for c in updated_delrecords.columns if c != "FileKey"]
)

# 3️⃣ Join on FileKey
final_target_rec = updated_add_records.join(
    updated_delrecords_renamed, on="FileKey", how="left"
)

# 4️⃣ Select only original columns from updated_add_records (31 columns)
final_target_rec = final_target_rec.select([F.col(c) for c in columns_to_keep])

# ✅ Check final schema
final_target_rec.show(5)


StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 17, Finished, Available, Finished)

+-------+----------+-------+--------+------+------+--------+--------+-------------+--------+-----------+-----+--------------+-------+----------------+----------------+--------------+-------------+--------------+------------------+---------+
|FileKey|LibraryKey|SiteKey|ObjectID|ItemID|SiteID|FileName|FilePath|FileExtension|FileType|ContentType|Owner|DataRedundancy|Version|IsCurrentVersion|SensitivityLabel|Classification|ComplianceTag|ApprovalStatus|IsSharedExternally|IsDeleted|
+-------+----------+-------+--------+------+------+--------+--------+-------------+--------+-----------+-----+--------------+-------+----------------+----------------+--------------+-------------+--------------+------------------+---------+
+-------+----------+-------+--------+------+------+--------+--------+-------------+--------+-----------+-----+--------------+-------+----------------+----------------+--------------+-------------+--------------+------------------+---------+
only showing top 0 rows

+-------+--

In [16]:
 # ===== WRITE PROCESS =====
try:
    # Write to Gold layer (PreGold_Reporting)
    print(f"Writing to Gold layer: {full_target_path}")

    full_target_path_new = full_target_path    
    # ## .rsplit('.', 1)[0]}_{datetime.now():%Y%m%d}.parquet"

    final_target_rec.write.mode("overwrite").option("compression", "snappy") \
    .parquet(f"{full_target_path_new}")

    # final_target.write.mode("overwrite").option("compression", "snappy") \
    # .parquet(f"{full_target_path.rsplit('.',1)[0]}_{datetime.now():%Y%m%d}.parquet")
    print(f"Successfully written to: {full_target_path_new}")
    
    # Verify the written file
    df_verify = spark.read.parquet(full_target_path_new)
    print(f"Verification - Target record count: {df_verify.count()}")
    
except Exception as e:
    print(f"Error in processing: {str(e)}")
    print("Please check:")
    print(f"1. Source file exists: {full_source_path}")
    print(f"2. Target path is accessible: {base_target_path}")

print("Process completed!")


StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 18, Finished, Available, Finished)

Writing to Gold layer: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files/PreGold_Reporting/Dim_Files.parquet
Successfully written to: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files/PreGold_Reporting/Dim_Files.parquet
Verification - Target record count: 2162
Process completed!


In [17]:
row_count = final_target_rec.count()
print(row_count)

StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 19, Finished, Available, Finished)

2162


In [18]:

# Ensure processing_successful is defined before this block
try:
    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Your ETL logic here
    # Example:
    # source_df = spark.read.format("delta").load(SOURCE_PATH)
    # transformed_df = source_df.filter("status = 'active'")
    # transformed_df.write.format("delta").mode("overwrite").save(TARGET_PATH)

    # Simulated metrics
    rows_read = source_df.count()
    rows_written = final_target_rec.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                    #  file_count=file_count,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, 7182826c-0b8b-460e-9ad7-52eaed00dafe, 20, Finished, Available, Finished)

🔄 Starting ETL processing for ntk_Sil2Gld_DimFiles V2...
✅ ntk_Sil2Gld_DimFiles V2 completed successfully (82s)
🎉 ntk_Sil2Gld_DimFiles V2 pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 2,160 → 2,162
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ntk_Sil2Gld_DimFiles V2:
+-----+------+---------+-------+---------------+--------+-----------+
|LogID|Status|StartTime|EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+------+---------+-------+---------------+--------+-----------+
+-----+------+---------+-------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ntk_Sil2Gld_DimFiles V2 logging completed!
